# Dynamics Transformer — perturbation × time → per-gene response

Physics-informed gene-transformer trained on **LINCS L1000**, held out **by perturbation**. **Needs a GPU runtime.**

Long cells now STREAM output live (you'll see each epoch's MSE + seconds). Training uses bf16 autocast (~2-3× faster on L4).

In [ ]:
# 1) GPU check + clone repo + deps
import torch, subprocess, os, sys
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE  ->  Runtime > Change runtime type > GPU')
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull'])
subprocess.run('pip install -q cmapPy h5py pandas openpyxl', shell=True)
print('cwd', os.getcwd())

In [ ]:
# 2) PHYSICS features: measured half-lives (CC-BY) + the EGF test course
import subprocess, sys
subprocess.run([sys.executable,'colab/fetch_dynamics_data.py'])
subprocess.run([sys.executable,'colab/fetch_timecourse.py'])

In [ ]:
# 3) LINCS L1000 -> training table (GSE92742 full 20 GB; skips re-download if already cached)
import subprocess, sys, os
os.environ['LINCS_GSE']='GSE92742'; os.environ['LINCS_MAX_SIGS']='120000'
subprocess.run([sys.executable,'colab/fetch_lincs.py'])   # streams live

In [ ]:
# 4) TRAIN (streams each epoch live). bf16 autocast. Tune with the env vars below.
import subprocess, sys, os
os.environ['DTF_EPOCHS']='10'; os.environ['DTF_DIM']='128'; os.environ['DTF_LAYERS']='3'; os.environ['DTF_BATCH']='256'
# OOM? set DTF_BATCH='96'.  Too slow? DTF_LAYERS='2' or DTF_DIM='96'.
subprocess.run([sys.executable,'colab/dynamics_transformer.py'])   # streams live

In [ ]:
# 5) STAGE-1 TEST: query the model at EGF's minute-timepoints -> does it BEAT 0.25?
import subprocess, sys
subprocess.run([sys.executable,'colab/eval_transformer_egf.py'])

In [ ]:
# 6) save checkpoint + evals to Drive
from google.colab import drive; drive.mount('/content/drive')
import shutil, os, json
d='/content/drive/MyDrive/virtual_cell_data/dynamics_transformer'; os.makedirs(d,exist_ok=True)
for f in ['dynamics_transformer.pt','dynamics_transformer_eval.json','eval_transformer_egf.json','lincs_train.npz']:
    p=f'outputs/orphan/{f}'
    if os.path.exists(p): shutil.copy(p,d); print('saved',f, os.path.getsize(p)//1048576,'MB')